# 01 - ModelFormatter baseline

Evaluate images → untouched VLM transcript → frozen text formatting model → task JSON on the fixed eight validation sequences.

The default local models are `C:\MODELS\HF\Qwen2.5-VL-3B-Instruct` for image transcription and `C:\MODELS\HF\Qwen2.5-1.5B-Instruct` for JSON formatting. Override them with `MANGA_VLM_MODEL_PATH` and `MANGA_FORMATTER_MODEL_PATH`, or edit the paths in the setup cell. Both models load local files only. The formatter runs on CPU by default; it never receives labels or images.

Run the cells in order from a fresh kernel. The VLM generates and saves all validation transcripts first; it is then unloaded before the frozen text formatter loads. Raw transcripts are kept in the evaluation output folder under `raw_transcripts/predictions.jsonl`.

To retry formatting in a fresh kernel without regenerating images, run only the setup and dataset cells, then the **Stage 2** cells below. Do not run the VLM loading or training cells for that retry.


In [1]:
from __future__ import annotations

import gc
import json
import os
import random
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

ROOT = next((path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (path / "Scripts" / "dataUtils.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Could not locate the repository root")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from Scripts import dataUtils as dU
from Scripts import trainValSplit
from Scripts.evaluator import calculate_metrics, evaluate_model, inspect_prediction, load_prediction_records, display_metric_summary, prediction_report

SEED = 42
MODEL_ROOT = Path(r"C:\MODELS\HF")
MODEL_PATH = Path(os.environ.get("MANGA_VLM_MODEL_PATH") or MODEL_ROOT / "Qwen2.5-VL-3B-Instruct").expanduser()
FORMATTER_MODEL_PATH = Path(os.environ.get("MANGA_FORMATTER_MODEL_PATH") or MODEL_ROOT / "Qwen2.5-1.5B-Instruct").expanduser()
for model_role, model_path, override_name in (
    ("VLM", MODEL_PATH, "MANGA_VLM_MODEL_PATH"),
    ("JSON formatter", FORMATTER_MODEL_PATH, "MANGA_FORMATTER_MODEL_PATH"),
): 
    if not model_path.is_dir():
        raise FileNotFoundError(f"{model_role} model folder not found: {model_path}. Download the model there or set {override_name} to its local folder.")
    print(f"{model_role} model path: {model_path}")
LABELS_PATH = ROOT / "TaskData" / "development" / "labels.jsonl"
IMAGE_ROOT = ROOT / "TaskData" / "development" / "images"
SPLIT_PATH = ROOT / "outputs" / "canonical_split_seed42.json"
OUTPUT_DIR = ROOT / "outputs" / "formatter_base_baseline"
MAX_NEW_TOKENS = 1024
MINIMUM_SIMILARITY = 0.50
REPETITION_PENALTY = 1.05

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"CUDA available: {torch.cuda.is_available()}")

from Scripts.modelFormatter import (FormatterSequenceDataset, FormatterPromptBuilder, ModelJSONFormatter, export_transcript_dataset, format_transcript, format_saved_transcripts, transcript_to_pages)
WORKFLOW_DIR = ROOT / "notebooks-ModelFormatter"
TRANSCRIPT_PROMPT = (WORKFLOW_DIR / "prompts" / "transcript.txt").read_text(encoding="utf-8").strip()
FORMATTER_PROMPT = (WORKFLOW_DIR / "prompts" / "formatter.txt").read_text(encoding="utf-8").strip()
FORMATTER_DEVICE = os.environ.get("MANGA_FORMATTER_DEVICE", "cpu")
FORMATTER_MAX_NEW_TOKENS = 2048

RAW_TRANSCRIPT_DIR = OUTPUT_DIR / "raw_transcripts"
TRANSCRIPT_PATH = RAW_TRANSCRIPT_DIR / "predictions.jsonl"


VLM model path: C:\MODELS\HF\Qwen2.5-VL-3B-Instruct
JSON formatter model path: C:\MODELS\HF\Qwen2.5-1.5B-Instruct
CUDA available: True


In [2]:
labels = dU.load_labels(LABELS_PATH)
all_ids = list(labels)
train_ids, validation_ids = trainValSplit.fit_seq(all_ids, v_size=0.10, random_state=SEED)
split = {"seed": SEED, "validation_fraction": 0.10, "train_ids": train_ids, "validation_ids": validation_ids}
SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)
if SPLIT_PATH.is_file():
    existing_split = json.loads(SPLIT_PATH.read_text(encoding="utf-8"))
    if existing_split != split:
        raise RuntimeError(f"Existing split differs from the seed-42 split: {SPLIT_PATH}")
else:
    SPLIT_PATH.write_text(json.dumps(split, indent=2), encoding="utf-8")

validation_dataset = FormatterSequenceDataset(LABELS_PATH, IMAGE_ROOT, validation_ids)
assert len(all_ids) == 80
assert len(train_ids) == 72 and len(validation_dataset) == 8
assert set(train_ids).isdisjoint(validation_ids)
display(pd.Series({"all sequences": len(all_ids), "training": len(train_ids), "validation": len(validation_dataset)}, name="count").to_frame())
display(pd.DataFrame({"validation_sequence_id": validation_ids}))

train_dataset = FormatterSequenceDataset(LABELS_PATH, IMAGE_ROOT, train_ids)
export_transcript_dataset(train_dataset, WORKFLOW_DIR / "datasets" / "train.jsonl")
export_transcript_dataset(validation_dataset, WORKFLOW_DIR / "datasets" / "validation.jsonl")
assert all(sample["transcript_target"] for sample in train_dataset)
print(train_dataset[0]["transcript_target"])

workflow_split_path = WORKFLOW_DIR / "datasets" / "split_seed42.json"
workflow_split_path.parent.mkdir(parents=True, exist_ok=True)
workflow_split_path.write_text(json.dumps(split, indent=2), encoding="utf-8")


,count
all sequences,80
training,72
validation,8


,validation_sequence_id
0,seq_d8bc3c6ac1bb6017
1,seq_2032620aa4e4ac7f
2,seq_d7af6c64ebc22bbe
3,seq_535e0754d805105b
4,seq_210721b0d576149f
5,seq_dce8be23c3281228
6,seq_1c1d4f5e71a2f554
7,seq_098e260f231aa789


PAGE 1
character1	?
character2	AH...
PAGE 2
character1	WH... W-WHAT'S WRONG, MIKAGE!?
character2	AH...
character2	OH, IT'S NOTHING!
character1	"NOTHING!?" KOMABA, DID YOU DO SOMETHING!?
character3	IT'S NOTHING!
character3	IT'S NONE OF YOUR BUSINESS!
character1	WAIT...
character1	HEY!!
character1	AH... UM.. ARE YOU ALL RIGHT?
character1	Um... Um... Um...
character2	SORRY, IT'S NOTHING.
character2	I'M FINE, REALLY!
character2	DON'T WORRY ABOUT IT!
PAGE 3
character1	WHAT THE HECK HAPPENED...?


2333

In [3]:
processor = AutoProcessor.from_pretrained(MODEL_PATH, local_files_only=True, min_pixels=128 * 28 * 28, max_pixels=256 * 28 * 28)
prompt_builder = FormatterPromptBuilder(TRANSCRIPT_PROMPT)
eval_collator = dU.MangaCollator(processor, process_vision_info, prompt_builder, train_mode=False, return_metadata=True)
sample_inputs, sample_metadata = eval_collator([validation_dataset[0]])
assert "labels" not in sample_inputs
assert sample_metadata[0]["target"] == validation_dataset[0]["target"]
print(f"Prompt tokens for first validation sequence: {sample_inputs['input_ids'].shape[1]}")
del sample_inputs, sample_metadata



experiment = {"pipeline": "images -> VLM transcript -> text model -> JSON", "vlm_model": str(MODEL_PATH), "formatter_model": str(FORMATTER_MODEL_PATH), "formatter_device": FORMATTER_DEVICE, "transcript_prompt": TRANSCRIPT_PROMPT, "formatter_prompt": FORMATTER_PROMPT, "split": split, "max_new_tokens": MAX_NEW_TOKENS, "formatter_max_new_tokens": FORMATTER_MAX_NEW_TOKENS, "repetition_penalty": REPETITION_PENALTY, "min_pixels": 128 * 28 * 28, "max_pixels": 256 * 28 * 28}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "experiment.json").write_text(json.dumps(experiment, indent=2), encoding="utf-8")


Prompt tokens for first validation sequence: 1371


6931

In [4]:
# Release a formatter left from an earlier run before loading the VLM.
globals().pop("formatter", None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for the 4-bit VLM baseline")
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_PATH, local_files_only=True, torch_dtype=torch.bfloat16, quantization_config=quantization_config, device_map={"": 0})
base_model.eval()
print(f"Loaded base model on {next(base_model.parameters()).device}")

W1001 19:28:21.879000 33624 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Loaded base model on cuda:0


## Preview a raw VLM transcript

This stage uses only the VLM. JSON formatting and scoring happen after all eight transcripts have been saved and the VLM has been unloaded. Change `SAMPLE_INDEX` to inspect another validation sample.

Speaker IDs must be generated by the VLM from the images. The preview validates the required page headers and speaker/text lines and displays the extracted character IDs. Invalid transcripts remain baseline failures; the text formatter cannot reconstruct missing speaker identity.


In [5]:
SAMPLE_INDEX = 0
sample = validation_dataset[SAMPLE_INDEX]
sample_inputs, sample_metadata = eval_collator([sample])
device = next(base_model.parameters()).device
sample_inputs = {name: value.to(device) if torch.is_tensor(value) else value for name, value in sample_inputs.items()}

with torch.inference_mode():
    generated = base_model.generate(
        **sample_inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        use_cache=True,
        repetition_penalty=REPETITION_PENALTY,
    )

prompt_width = sample_inputs["input_ids"].shape[1]
prediction = processor.decode(generated[0, prompt_width:], skip_special_tokens=True, clean_up_tokenization_spaces=False)
raw_transcript = prediction
print("RAW VLM TRANSCRIPT:\n", raw_transcript)
preview_generated_tokens = generated.shape[1] - prompt_width
print(f"Generated tokens: {preview_generated_tokens}/{MAX_NEW_TOKENS}")
if preview_generated_tokens >= MAX_NEW_TOKENS:
    print("WARNING: Token limit reached; the transcript may be incomplete or stuck repeating.")

# Inspect only the generated transcript; never fill missing speakers from targets.
try:
    preview_pages = transcript_to_pages(raw_transcript)
except ValueError as error:
    print(f"INVALID VLM TRANSCRIPT: {error}")
    print("Expected PAGE 1/2/3 headers and speaker<TAB>text for every item.")
    print("Missing speaker IDs must be generated from the images; Stage 2 cannot infer them.")
else:
    preview_items = [
        {"page": page_number, **item}
        for page_number, page in enumerate(preview_pages, 1)
        for item in page
    ]
    character_ids = sorted(
        {item["speaker"] for item in preview_items if item["speaker"].startswith("character")},
        key=lambda speaker: int(speaker.removeprefix("character")),
    )
    print("VALID TRANSCRIPT FORMAT (speaker attribution still requires visual review)")
    print("Identified characters:", ", ".join(character_ids) or "none")
    print("UNKNOWN items:", sum(item["speaker"] == "UNKNOWN" for item in preview_items))
    display(pd.DataFrame(preview_items, columns=["page", "speaker", "text"]))

del sample_inputs, sample_metadata, generated
gc.collect()
torch.cuda.empty_cache()


RAW VLM TRANSCRIPT:
 PAGE 1
character1	OH...
character2	GOD!
NARRATION	MARCH 4, 1985, JEROME PARK, THE BRONX, NYC
PAGE 2
character1	CALL AN AMBULANCE, ARBU!
NARRATION	UNKNOWN
PAGE 3
character1	WHAD'YA THINK, JENKINS?
character2	SUICIDE?
NARRATION	UNKNOWN
Generated tokens: 94/1024
VALID TRANSCRIPT FORMAT (speaker attribution still requires visual review)
Identified characters: character1, character2
UNKNOWN items: 0


,page,speaker,text
0,1,character1,OH...
1,1,character2,GOD!
2,1,NARRATION,"MARCH 4, 1985, JEROME PARK, THE BRONX, NYC"
3,2,character1,"CALL AN AMBULANCE, ARBU!"
4,2,NARRATION,UNKNOWN
5,3,character1,"WHAD'YA THINK, JENKINS?"
6,3,character2,SUICIDE?
7,3,NARRATION,UNKNOWN


In [6]:
# Stage 1: generate every VLM transcript without loading the formatter.
evaluate_model(
    model=base_model,
    processor=processor,
    dataset=validation_dataset,
    collator=eval_collator,
    batch_size=1,
    max_new_tokens=MAX_NEW_TOKENS,
    minimum_similarity=MINIMUM_SIMILARITY,
    repetition_penalty=REPETITION_PENALTY,
    output_dir=RAW_TRANSCRIPT_DIR,
    generation_only=True,
)
print(f"Saved raw VLM transcripts to {TRANSCRIPT_PATH}")


Evaluated 1/8 sequences
Evaluated 2/8 sequences
Evaluated 3/8 sequences
Evaluated 4/8 sequences
Evaluated 5/8 sequences
Evaluated 6/8 sequences
Evaluated 7/8 sequences
Evaluated 8/8 sequences
Saved raw VLM transcripts to C:\Users\ESHAAN\BYTE\Applied-Task\manga_vlm_crashcourse\outputs\formatter_base_baseline\raw_transcripts\predictions.jsonl


In [22]:
import json
from pathlib import Path
from pprint import pprint

path = Path("outputs/formatter_base_baseline/raw_transcripts/predictions.jsonl")
if not path.exists():  # When the notebook runs from a subfolder
    path = Path("..") / path

records = [
    json.loads(line)
    for line in path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

print(f"Records: {len(records)}")
print("Available keys:", sorted({key for record in records for key in record}))

ROW = 2
KEY = "prediction"  # e.g. target, sequence_id, image_input_diagnostics

value = records[ROW][KEY]
seq = records[ROW]['sequence_id']
print(seq)
print(f"\nRecord {ROW} | {KEY}\n")
if isinstance(value, str):
    print(value)
else:
    pprint(value, sort_dicts=False)

Records: 8
Available keys: ['generated_tokens', 'hit_max_new_tokens', 'image_input_diagnostics', 'image_paths', 'prediction', 'reached_eos', 'sequence_id', 'suspected_repetition_loop', 'target']
seq_d7af6c64ebc22bbe

Record 2 | prediction

PAGE 1
character1	visible dialogue
character2	another person's dialogue
NARRATION	visible narration
character1	POCHITA!
character2	HMM?
character1	I really hate you Devil Hunters!
character2	WHY DID THEY WANT A BETTER LIFE?
character1	AS LONG AS I HAD POCHITA THAT WAS ENOUGH.
character2	YOU GUYS! EAT HIM UP!
character1	THERE'S NOTHING WRONG WITH THAT.
character2	THERE'S NOTHING WRONG WITH THAT, BUT...
NARRATION	When they were already so well off?

 PAGE 2
character1	visible dialogue
character2	another person's dialogue
NARRATION	As long as I had Pochita that was enough.
character1	I get it. Everyone's got their own dreams.
character2	There's nothing wrong with that, but...

 PAGE 3
NARRATION	visible narration
character1	There's nothing wrong with tha

## Stage 2: unload the VLM and format saved transcripts

This cell reads the saved transcripts from disk. The formatter receives only transcript text; targets and images remain evaluation metadata. Training state is also released in the LoRA notebook. Formatting failures remain in the scored outputs.


In [7]:
# Confirm all holdout transcripts are saved before releasing the VLM.
saved_transcripts = load_prediction_records(TRANSCRIPT_PATH)
if sorted(row["sequence_id"] for row in saved_transcripts) != sorted(validation_ids):
    raise RuntimeError("Saved transcripts do not match the complete validation split; rerun Stage 1")
del saved_transcripts

# Drop every notebook reference to the VLM and training state before formatting.
for name in ("trainer", "model", "base_model", "processor", "train_collator",
             "eval_collator", "sample_inputs", "sample_metadata", "generated",
             "formatter"):
    globals().pop(name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("VLM and training state released; loading the text formatter next.")

formatter = ModelJSONFormatter(
    FORMATTER_MODEL_PATH, FORMATTER_PROMPT,
    device=FORMATTER_DEVICE, max_new_tokens=FORMATTER_MAX_NEW_TOKENS,
)
try:
    baseline_metrics = format_saved_transcripts(
        TRANSCRIPT_PATH, formatter, OUTPUT_DIR,
        minimum_similarity=MINIMUM_SIMILARITY,
    )
finally:
    del formatter
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print(f"Saved formatted JSON predictions and metrics to {OUTPUT_DIR}")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


VLM and training state released; loading the text formatter next.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Formatted 1/8 sequences
Formatted 2/8 sequences
Formatted 3/8 sequences
Formatted 4/8 sequences
Formatted 5/8 sequences
Formatted 6/8 sequences
Formatted 7/8 sequences
Formatted 8/8 sequences
Saved formatted JSON predictions and metrics to C:\Users\ESHAAN\BYTE\Applied-Task\manga_vlm_crashcourse\outputs\formatter_base_baseline


Compare the model formatter with a tolerant deterministic parser on the same transcripts. The model remains the main pipeline; parser predictions and metrics are saved separately.


In [ ]:
# Compare serialization on the same saved VLM transcripts; no model reload.
parser_metrics = format_saved_transcripts(
    TRANSCRIPT_PATH, lambda text: format_transcript(text, tolerant=True),
    OUTPUT_DIR / "tolerant_parser", minimum_similarity=MINIMUM_SIMILARITY,
)
display(pd.DataFrame([
    {"formatter": "text model", "schema_compliance_rate": baseline_metrics["schema_compliance_rate"]},
    {"formatter": "tolerant parser", "schema_compliance_rate": parser_metrics["schema_compliance_rate"]},
]))
# Full task metrics and per-sequence failures are saved separately under tolerant_parser.


In [8]:
SAMPLE_INDEX = globals().get("SAMPLE_INDEX", 0)
single_record = load_prediction_records(OUTPUT_DIR / "predictions.jsonl")[SAMPLE_INDEX]
print(f"SEQUENCE: {single_record['sequence_id']}")
print("RAW VLM TRANSCRIPT:\n", single_record["raw_transcript"])
print("RAW FORMATTER RESPONSE:\n", single_record["raw_formatter_prediction"])
print("FORMATTER ERROR:", single_record["formatter_error"])
print("TARGET:\n", json.dumps(single_record["target"], ensure_ascii=False, indent=2))
single_metrics = calculate_metrics([single_record], minimum_similarity=MINIMUM_SIMILARITY)
single_report = prediction_report(single_record, minimum_similarity=MINIMUM_SIMILARITY)
display_metric_summary(single_metrics, report=single_report)


SEQUENCE: seq_d8bc3c6ac1bb6017
RAW VLM TRANSCRIPT:
 PAGE 1
character1	OH...
character2	GOD!
NARRATION	MARCH 4, 1985, JEROME PARK, THE BRONX, NYC
PAGE 2
character1	CALL AN AMBULANCE, ARBU!
NARRATION	UNKNOWN
PAGE 3
character1	WHAD'YA THINK, JENKINS?
character2	SUICIDE?
NARRATION	UNKNOWN
RAW FORMATTER RESPONSE:
 [["character1", "OH..."], [], ["character1", "WHAD'YA THINK, JENKINS?"]]

null
FORMATTER ERROR: Formatting model returned invalid JSON: Extra data: line 3 column 1 (char 74)
TARGET:
 [
  [
    {
      "speaker": "NARRATION",
      "text": "MARCH 4, 1985. JEROME PARK, THE BRONX, NYC"
    }
  ],
  [
    {
      "speaker": "character1",
      "text": "OH... GOD!"
    },
    {
      "speaker": "character2",
      "text": "M-MA'AM!?"
    },
    {
      "speaker": "character1",
      "text": "JERRY..."
    },
    {
      "speaker": "character1",
      "text": "JERRY!!!"
    },
    {
      "speaker": "character1",
      "text": "C-CALL... CALL AN AMBU... AMBU..."
    }
  ],
  [
    {
   

,Result,Value,What it tells us
0,Format valid?,No,0/1 outputs usable; check page-count diagnostics
1,Text error rate (lower is better),N/A — invalid output,0 reference characters on valid outputs
2,Dialogue F1,N/A — invalid output,"0 matched, 0 missing, 0 extra; 9 target lines ..."
3,Speaker accuracy,N/A — invalid output,0/0 matched dialogues correct after character-...


Format: JSON parse failed: Expecting value: line 1 column 1 (char 0)


In [9]:
display_metric_summary(baseline_metrics)

records = load_prediction_records(OUTPUT_DIR / "predictions.jsonl")
per_sequence = []
for record in records:
    metrics = calculate_metrics([record], minimum_similarity=MINIMUM_SIMILARITY)
    per_sequence.append({"sequence_id": record["sequence_id"], "format_valid": bool(metrics["schema_compliance_rate"]), "text_error_rate": metrics["cer_valid_outputs"], "failure_penalty_cer": metrics["cer_including_failures_as_deletions"], "missing_dialogues": metrics["support"]["missing_dialogues"] if metrics["cer_evaluated_sequences"] else None, "extra_dialogues": metrics["support"]["extra_dialogues"] if metrics["cer_evaluated_sequences"] else None, "speaker_accuracy": metrics["matched_speaker_accuracy"], "dialogue_f1": metrics["dialogue_text_match_f1"], "exact_match": metrics["sequence_exact_match"], "generated_tokens": record["generated_tokens"], "hit_max_new_tokens": record["hit_max_new_tokens"], "repetition_loop": record["suspected_repetition_loop"]})
review = pd.DataFrame(per_sequence).sort_values(["format_valid", "failure_penalty_cer"], ascending=[True, False]).reset_index(drop=True)
display(review.style.format(na_rep="N/A"))
_ = inspect_prediction(OUTPUT_DIR / "predictions.jsonl", sequence_id=review.iloc[0]["sequence_id"], minimum_similarity=MINIMUM_SIMILARITY)

,Result,Value,What it tells us
0,Format valid?,0.000,0/8 outputs usable; check page-count diagnostics
1,Text error rate (lower is better),N/A — invalid output,0 reference characters on valid outputs
2,Dialogue F1,N/A — invalid output,"0 matched, 0 missing, 0 extra; 139 target line..."
3,Speaker accuracy,N/A — invalid output,0/0 matched dialogues correct after character-...


,sequence_id,format_valid,text_error_rate,failure_penalty_cer,missing_dialogues,extra_dialogues,speaker_accuracy,dialogue_f1,exact_match,generated_tokens,hit_max_new_tokens,repetition_loop
0,seq_d8bc3c6ac1bb6017,False,N/A,1.000000,N/A,N/A,N/A,N/A,0.000000,94,False,False
1,seq_2032620aa4e4ac7f,False,N/A,1.000000,N/A,N/A,N/A,N/A,0.000000,108,False,False
2,seq_d7af6c64ebc22bbe,False,N/A,1.000000,N/A,N/A,N/A,N/A,0.000000,221,False,False
3,seq_535e0754d805105b,False,N/A,1.000000,N/A,N/A,N/A,N/A,0.000000,1024,True,False
4,seq_210721b0d576149f,False,N/A,1.000000,N/A,N/A,N/A,N/A,0.000000,165,False,False
5,seq_dce8be23c3281228,False,N/A,1.000000,N/A,N/A,N/A,N/A,0.000000,1024,True,False
6,seq_1c1d4f5e71a2f554,False,N/A,1.000000,N/A,N/A,N/A,N/A,0.000000,1024,True,False
7,seq_098e260f231aa789,False,N/A,1.000000,N/A,N/A,N/A,N/A,0.000000,1024,True,False


SEQUENCE
seq_d8bc3c6ac1bb6017

TARGET
[
  [
    {
      "speaker": "NARRATION",
      "text": "MARCH 4, 1985. JEROME PARK, THE BRONX, NYC"
    }
  ],
  [
    {
      "speaker": "character1",
      "text": "OH... GOD!"
    },
    {
      "speaker": "character2",
      "text": "M-MA'AM!?"
    },
    {
      "speaker": "character1",
      "text": "JERRY..."
    },
    {
      "speaker": "character1",
      "text": "JERRY!!!"
    },
    {
      "speaker": "character1",
      "text": "C-CALL... CALL AN AMBU... AMBU..."
    }
  ],
  [
    {
      "speaker": "character3",
      "text": "WHADD'YA THINK, JENKINS?"
    },
    {
      "speaker": "character4",
      "text": "SUICIDE?"
    },
    {
      "speaker": "character4",
      "text": "...PARDON ME."
    }
  ]
]

PREDICTION
INVALID JSON:


RESULT
Format valid?: No (0/1 outputs usable; check page-count diagnostics)
Text error rate (lower is better): N/A — invalid output (0 reference characters on valid outputs)
Dialogue F1: N/A — invalid out

In [10]:
records = load_prediction_records(OUTPUT_DIR / "predictions.jsonl")

for record in records:
    print(
        record["sequence_id"],
        "tokens:", record["generated_tokens"],
        "hit limit:", record["hit_max_new_tokens"],
        "EOS:", record["reached_eos"],
    )

seq_d8bc3c6ac1bb6017 tokens: 94 hit limit: False EOS: True
seq_2032620aa4e4ac7f tokens: 108 hit limit: False EOS: True
seq_d7af6c64ebc22bbe tokens: 221 hit limit: False EOS: True
seq_535e0754d805105b tokens: 1024 hit limit: True EOS: False
seq_210721b0d576149f tokens: 165 hit limit: False EOS: True
seq_dce8be23c3281228 tokens: 1024 hit limit: True EOS: False
seq_1c1d4f5e71a2f554 tokens: 1024 hit limit: True EOS: False
seq_098e260f231aa789 tokens: 1024 hit limit: True EOS: False


In [ ]:
import json
from pathlib import Path
from pprint import pprint

path = Path("outputs/formatter_base_baseline/predictions.jsonl")
if not path.exists():  # When the notebook runs from a subfolder
    path = Path("..") / path

records = [
    json.loads(line)
    for line in path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

print(f"Records: {len(records)}")
print("Available keys:", sorted({key for record in records for key in record}))

ROW = 1
KEY = "formatter_error"  # e.g. target, sequence_id, image_input_diagnostics

value = records[ROW][KEY]
seq = records[ROW]['sequence_id']
print(seq)
print(f"\nRecord {ROW} | {KEY}\n")
if isinstance(value, str):
    print(value)
else:
    pprint(value, sort_dicts=False)

Records: 8
Available keys: ['formatter_error', 'generated_tokens', 'hit_max_new_tokens', 'image_input_diagnostics', 'image_paths', 'prediction', 'raw_formatter_prediction', 'raw_transcript', 'reached_eos', 'sequence_id', 'suspected_repetition_loop', 'target']
seq_2032620aa4e4ac7f

Record 1 | target

[[{'speaker': 'character1',
   'text': 'HUH? DOES KOMABA SEEM DIFFERENT THAN USUAL?'},
  {'speaker': 'character2',
   'text': "HE'S DOING REALLY WELL. IT LOOKS LIKE HE'S BEEN WORKING HARD FOR "
           'THE FALL TOURNAMENT.'},
  {'speaker': 'UNKNOWN', 'text': "THAT'S THE WAY!"},
  {'speaker': 'character1', 'text': "NO, THAT'S NOT IT. IT'S LIKE HE'S..."},
  {'speaker': 'character1', 'text': 'OUT FOR BLOOD.'}],
 [],
 [{'speaker': 'UNKNOWN', 'text': 'WOAH!!'},
  {'speaker': 'character3', 'text': 'OKAWA-SENPAI, YOU LOOK SO COOL!'},
  {'speaker': 'character4', 'text': 'RIGHT HERE! THIS IS WHERE IT HAPPENS!'},
  {'speaker': 'UNKNOWN', 'text': 'SO CLOSE!!'},
  {'speaker': 'character5',
   'text

## Official task score: primary experiment benchmark

Use the unmodified `TaskData/score.py` to judge text order and balanced text/speaker quality on the fixed holdout. Inspect invalid outputs and missing sequences before accepting an improvement.


In [11]:
from Scripts.taskScore import score_saved_predictions

official_score = score_saved_predictions(OUTPUT_DIR / "predictions.jsonl", LABELS_PATH, validation_ids, OUTPUT_DIR / "task_score")
display(pd.Series(official_score["macro"], name="official_base_score").to_frame())
print("Missing or invalid sequences:", official_score["missing_sequences"])


,official_base_score
text_order_score,0.0
balanced_joint_f1,0.0
speaker_accuracy_on_matched,0.0
joint_f1,0.0
matched_token_coverage,0.0


Missing or invalid sequences: ['seq_098e260f231aa789', 'seq_1c1d4f5e71a2f554', 'seq_2032620aa4e4ac7f', 'seq_210721b0d576149f', 'seq_535e0754d805105b', 'seq_d7af6c64ebc22bbe', 'seq_d8bc3c6ac1bb6017', 'seq_dce8be23c3281228']
